# Homework5: Exploration of Heart Disease Data (Use UCI Dataset)
## Part 1: Confidence Intervals for a Personalized Subgroup
    1. Define a subgroup of patients based on age, sex, chest pain type, or another variable of your choice.
    2. For this subgroup:
        - Randomly select three different sample sizes (e.g., small, medium, large).
        - Calculate the proportion of patients with heart disease.
        - Construct 95% confidence intervals.
    3. Questions:
        - How does interval width change with sample size?
        - Compare your subgroup’s proportion to the overall population.
        - Write 2–3 sentences explaining why your subgroup may differ from the population.
        
## Part 2: Hypothesis Tests 

### Choose two tests that were not done in class. For each:

    - State your research question and define your null and alternative hypotheses.
    - Select an appropriate test (two-proportion, one-sample t-test, two-sample t-test).
    - Compute the test and report the statistic and p-value.
    - Visualize the data appropriately (bar chart, histogram, or boxplot).
    - Write 2–3 sentences interpreting your results.
    
## Part 3: Type I and Type II Errors
### For one of your tests above, describe:
    - What a Type I error would mean in your context.
    - What a Type II error would mean.
    - Explain which error would be more critical for patient care or medical research.

## Part 4: Sample Size and Power Planning
    - Pick one of your tests.
    - Assume α = 0.05 and desired power between 0.8–0.9.
    - Estimate effect size from your data or justify a hypothetical one.
    - Compute the required sample size.
    - Optional: Generate a power curve showing power vs sample size for your chosen effect size.
    - Write 2–3 sentences interpreting how sample size, effect size, and power interact.

Import Packages

In [34]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower, TTestIndPower, TTestPower

df = pd.read_csv('Heartdisease.csv')

#create a new column with yes (1) or no (0) status of heart disease 
df['heart_disease'] = [ 1 if target >0 else 0 for target in df['target'] ]

#df.info()

Part 1: Confidence Intervals for a Personalized Subgroup - Females

In [38]:
females = df[df['sex'] == 0]

# Function to compute CI for a given sample size
def sample_ci(females, n, alpha=0.05):
    sample = females.sample(n, random_state=42)
    x = sample['heart_disease'].sum()  # number of women with heart disease in the sample
    p_hat = x / n
    ci_low, ci_upp = proportion_confint(count=x, nobs=n, alpha=alpha, method='wilson') #wilson can be used for small sample sizes. 
    return p_hat, ci_low, ci_upp

# Different sample sizes
sample_sizes = [10, 30, 80]
ci_results = []

for n in sample_sizes:
    p_hat, low, up = sample_ci(females, n)
    ci_results.append({'n': n, 'p_hat': p_hat, 'CI_low': low, 'CI_high': up, 'width': up-low})

ci_df = pd.DataFrame(ci_results)
ci_df




,n,p_hat,CI_low,CI_high,width
0,10,0.200000,0.056682,0.509838,0.453155
1,30,0.233333,0.117924,0.409283,0.291359
2,80,0.287500,0.199869,0.394603,0.194734


Part 1 Questions:

As the sample size gets larger, the interval width gets smaller. The larger the sample, the more accurate the interval is for a given confidence level. 

This proportion of around 25% is lower than the overall population. This is because men have heart disease at higher rates than women. In addition, this data set contained significantly more male participants than female, which will skew the population proportion higher. 



Part 2: Hypothesis Tests

Test 1: Do patients with diabetes have higher rates of heart disease?
H0: There is no difference in rates of heart disease between patients with and without diabetes. 
Ha: There is a significant difference in the rates of heart disease in patients with diabetes. 

In [24]:
# Two-Proportion Z-Test 

diabetic = df[df['fbs'] == 1]['heart_disease']
non_diabetic = df[df['fbs'] == 0]['heart_disease']

#count the number of diabetics and non-diabetics with heart disease, then give the count of all diabetics and non-diabetics. 
count = np.array([diabetic.sum(), non_diabetic.sum()])
nobs = np.array([diabetic.shape[0], non_diabetic.shape[0]])

#calculate the z-statistic and p-value using Z-Test 
stat, pval = proportions_ztest(count, nobs)
print(f"Z-Statistic: {stat: .3f}, p-value: {pval: .4f}")





Z-Statistic:  0.055, p-value:  0.9565


A p-value of .9565 says that our datasets are nearly identical, so we can strongly accept the null hypothesis. There is no significant difference in the rates of heart disease between those with diabetes and those without. 

Test 2: Is the average age of patients with heart disease different from 65 years old?
H0: The average age of patients with heart disease is 65.
Ha: The average age of patients with heart disease is different from 65. 

In [27]:
# One sample T-test
mu0 = 65
hd_age = df[df['heart_disease'] == 1]['age']

t_stat, p_value = st.ttest_1samp(hd_age, mu0)
print(f"T-Statistic: {t_stat: .3f}, p-value: {p_value: .4f}")

T-Statistic: -12.210, p-value:  0.0000


A p-value of approximately zero tells us we can reject the null hypothesis, the average of patients with heart disease is not 65 years old. The t-stat tells us that the actually mean age is closer to 53 years old. 

Part 3: Type I and Type II Errors

In my first test about diabetes, a Type I error would be if there was no difference between diabetic and non-diabetic rates of heart disease, but I concluded that there was a differnce. The true null would be incorrectly rejected. A Type II error for this test would be if there was a difference in the rates of heart disease between the two groups, but I incorrectly found that there was no difference. I would have failed to reject a false null. 

In my view, a Type II error would be more significant. If diabetes did make you more likely to get heart disease and I claimed the opposite, this would have adverse health effects. Diabetics might not get the extra attention paid to their heart that they would need.  A Type I error would also be bad, but in that case it would cause people to try not to get diabetes, which they should be doing anyway. 

Part 4: Sample Size and Power Planning 

In [32]:
# Effect Size and Sample Size for Test 2, a one sample t-test. 

mu0 = 65
effect_size = (hd_age.mean()-mu0) /  hd_age.std()
alpha = .05 
power = .8

analysis = TTestPower()
required_n = analysis.solve_power(effect_size = effect_size, alpha = alpha, power = power, alternative = 'larger')

print(f"Effect Size = {effect_size:.3f}")
print(f"Required sample size = {np.ceil(required_n)}")




Effect Size = -1.043
Required sample size = [10.]


C:\Users\obruc\anaconda3\Lib\site-packages\statsmodels\stats\power.py:524: ConvergenceWarning: 
Failed to converge on a solution.

  warnings.warn(convergence_doc, ConvergenceWarning)


Larger sample sizes increase power, and allow you to detect smaller effects. Larger effect sizes lead to increased power, requiring smaller sample sizes. Higher power requires smaller sample sizes, and allows you to detect smaller effect sizes.  